### STEP 0 Load raw data

In [1]:
import pandas as pd
import os

# -------------------------------
# 1) ตั้งค่าให้ Pandas แสดงผลอ่านง่าย
# -------------------------------
pd.set_option('display.max_columns', None)   # แสดงทุกคอลัมน์
pd.set_option('display.width', 200)          # ความกว้างตาราง
pd.set_option('display.max_colwidth', 20)    # ความกว้างข้อความต่อคอลัมน์
pd.set_option('display.float_format', '{:.4f}'.format)  # ทศนิยมสวย ๆ

# -------------------------------
# 2) path ไปยังไฟล์ CSV
# -------------------------------
file_path = r"C:\punpun\Big data\US-Traffic-Accident-BigData-Analysis\data\raw\US_Accidents_March23.csv"

# -------------------------------
# 3) โหลดข้อมูล
# -------------------------------
if os.path.exists(file_path):
    print("พบไฟล์แล้ว กำลังโหลดข้อมูล...\n")
    
    df = pd.read_csv(file_path)
    
    print(f"โหลดสำเร็จ! จำนวนข้อมูล: {len(df):,} แถว")
    print(f"จำนวนคอลัมน์: {df.shape[1]}\n")
    
    # แสดง 5 แถวแรก แบบเห็นครบทุกคอลัมน์
    display(df.head())
else:
    print(f"ไม่พบไฟล์ที่: {os.path.abspath(file_path)}")


พบไฟล์แล้ว กำลังโหลดข้อมูล...

โหลดสำเร็จ! จำนวนข้อมูล: 7,728,394 แถว
จำนวนคอลัมน์: 46



,ID,Source,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,End_Lat,End_Lng,Distance(mi),Description,Street,City,County,State,Zipcode,Country,Timezone,Airport_Code,Weather_Timestamp,Temperature(F),Wind_Chill(F),Humidity(%),Pressure(in),Visibility(mi),Wind_Direction,Wind_Speed(mph),Precipitation(in),Weather_Condition,Amenity,Bump,Crossing,Give_Way,Junction,No_Exit,Railway,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop,Sunrise_Sunset,Civil_Twilight,Nautical_Twilight,Astronomical_Twilight
0,A-1,Source2,3,2016-02-08 05:46:00,2016-02-08 11:00:00,39.8651,-84.0587,NaN,NaN,0.0100,Right lane block...,I-70 E,Dayton,Montgomery,OH,45424,US,US/Eastern,KFFO,2016-02-08 05:58:00,36.9000,NaN,91.0000,29.6800,10.0000,Calm,NaN,0.0200,Light Rain,False,False,False,False,False,False,False,False,False,False,False,False,False,Night,Night,Night,Night
1,A-2,Source2,2,2016-02-08 06:07:59,2016-02-08 06:37:59,39.9281,-82.8312,NaN,NaN,0.0100,Accident on Bric...,Brice Rd,Reynoldsburg,Franklin,OH,43068-3402,US,US/Eastern,KCMH,2016-02-08 05:51:00,37.9000,NaN,100.0000,29.6500,10.0000,Calm,NaN,0.0000,Light Rain,False,False,False,False,False,False,False,False,False,False,False,False,False,Night,Night,Night,Day
2,A-3,Source2,2,2016-02-08 06:49:27,2016-02-08 07:19:27,39.0631,-84.0326,NaN,NaN,0.0100,Accident on OH-3...,State Route 32,Williamsburg,Clermont,OH,45176,US,US/Eastern,KI69,2016-02-08 06:56:00,36.0000,33.3000,100.0000,29.6700,10.0000,SW,3.5000,NaN,Overcast,False,False,False,False,False,False,False,False,False,False,False,True,False,Night,Night,Day,Day
3,A-4,Source2,3,2016-02-08 07:23:34,2016-02-08 07:53:34,39.7478,-84.2056,NaN,NaN,0.0100,Accident on I-75...,I-75 S,Dayton,Montgomery,OH,45417,US,US/Eastern,KDAY,2016-02-08 07:38:00,35.1000,31.0000,96.0000,29.6400,9.0000,SW,4.6000,NaN,Mostly Cloudy,False,False,False,False,False,False,False,False,False,False,False,False,False,Night,Day,Day,Day
4,A-5,Source2,2,2016-02-08 07:39:07,2016-02-08 08:09:07,39.6278,-84.1884,NaN,NaN,0.0100,Accident on McEw...,Miamisburg Cente...,Dayton,Montgomery,OH,45459,US,US/Eastern,KMGY,2016-02-08 07:53:00,36.0000,33.3000,89.0000,29.6500,6.0000,SW,3.5000,NaN,Mostly Cloudy,False,False,False,False,False,False,False,False,False,False,False,True,False,Day,Day,Day,Day


### Correlation 

In [2]:
date_cols = df.select_dtypes(include=['datetime64']).columns
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
cat_cols = df.select_dtypes(include=['object', 'string']).columns
bool_cols = df.select_dtypes(include=['bool']).columns

# แปลง datetime สำคัญ
df['End_Time'] = pd.to_datetime(df['End_Time'], errors='coerce')
df['Weather_Timestamp'] = pd.to_datetime(df['Weather_Timestamp'], errors='coerce')


### STEP 1 Parse datetime

In [3]:
datetime_cols = ['Start_Time', 'End_Time', 'Weather_Timestamp']

for col in datetime_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

df[datetime_cols].dtypes


Start_Time           datetime64[us]
End_Time             datetime64[us]
Weather_Timestamp    datetime64[us]
dtype: object

### STEP 2 : Recreate Time Features 

* โดยจะไม่ดึงจาก EDA — สร้างใหม่ให้ deterministic

In [4]:
df['Start_Hour'] = df['Start_Time'].dt.hour
df['Start_Weekday'] = df['Start_Time'].dt.weekday
df['Start_Month'] = df['Start_Time'].dt.month

df['End_Hour'] = df['End_Time'].dt.hour
df['End_Weekday'] = df['End_Time'].dt.weekday

df['Is_Weekend'] = df['Start_Weekday'].isin([5, 6]).astype(int)

df['Accident_Duration_Min'] = (
    df['End_Time'] - df['Start_Time']
).dt.total_seconds() / 60


### STEP 3 : Define Column Groups
* นี่คือ “สัญญา” ระหว่างคุณกับโมเดล

In [5]:
geo_cols = ['Start_Lat', 'Start_Lng', 'End_Lat', 'End_Lng']

weather_numeric = [
    'Temperature(F)', 'Humidity(%)', 'Pressure(in)',
    'Visibility(mi)', 'Wind_Speed(mph)',
    'Precipitation(in)', 'Wind_Chill(F)'
]

time_cols = [
    'Start_Hour', 'Start_Weekday', 'Start_Month',
    'End_Hour', 'End_Weekday', 'Accident_Duration_Min'
]

categorical_cols = [
    'Severity', 'State', 'Weather_Condition',
    'Sunrise_Sunset'
]

boolean_cols = [
    'Amenity', 'Bump', 'Crossing', 'Give_Way',
    'Junction', 'No_Exit', 'Railway', 'Roundabout',
    'Station', 'Stop', 'Traffic_Calming',
    'Traffic_Signal', 'Turning_Loop'
]


แปลงข้อมูลเวลาและสร้างคอลัมน์ช่วยวิเคราะห์

In [6]:
# 1. จัดการ Datetime ให้เรียบร้อยก่อน (แปลง String -> Datetime)
# ถ้าเจอรูปแบบผิด errors='coerce' จะเปลี่ยนเป็น NaT ให้เอง
datetime_cols = ['Start_Time', 'End_Time', 'Weather_Timestamp']
for col in datetime_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# 2. [ย้ายมาตรงนี้] ลบแถวที่ Start_Time เป็น NaT ทิ้ง
# ขั้นตอนนี้จะกรองออกหมดเกลี้ยง ทั้งค่าว่างเดิมและค่าที่แปลงไม่ได้
df = df.dropna(subset=['Start_Time'])
print(f"จำนวนข้อมูลหลังลบ Start_Time ที่เป็นค่าว่าง/เสีย: {len(df):,} แถว")

# 3. สร้าง Feature (ทำหลังลบ มั่นใจได้ว่าไม่มี Error แน่นอน)
df['Start_Month'] = df['Start_Time'].dt.month
df['Start_Hour'] = df['Start_Time'].dt.hour
df['Start_Weekday'] = df['Start_Time'].dt.weekday
df['Is_Weekend'] = df['Start_Weekday'].isin([5, 6]).astype(int)

# 4. คำนวณระยะเวลา (Duration)
df['Accident_Duration_Min'] = (df['End_Time'] - df['Start_Time']).dt.total_seconds() / 60
# (เสริม) อย่าลืมเติมค่าว่างให้ Duration เผื่อ End_Time หาย
df['Accident_Duration_Min'] = df['Accident_Duration_Min'].fillna(df['Accident_Duration_Min'].median())

จำนวนข้อมูลหลังลบ Start_Time ที่เป็นค่าว่าง/เสีย: 6,985,228 แถว


จัดการ Missing Value (จุดวิกฤต)

In [7]:
# 1. จัดการ Categorical ให้เรียบร้อย
df['Weather_Condition'] = df['Weather_Condition'].fillna('Unknown')
df['Sunrise_Sunset'] = df['Sunrise_Sunset'].fillna('Unknown')

# 2. เติมค่า Weather Numeric โดยอิงตามรัฐ (State) และเดือน (Month) เพื่อความสมจริง
weather_num = ['Temperature(F)', 'Wind_Chill(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)', 'Wind_Speed(mph)']

for col in weather_num:
    # เติมด้วยค่ากลางของรัฐนั้นในเดือนนั้น (เช่น อุณหภูมิ NY ในเดือนมกราคม)
    df[col] = df.groupby(['State', 'Start_Month'])[col].transform(lambda x: x.fillna(x.median()))
    # ถ้ายังว่าง ให้ใช้ค่ากลางของรัฐ
    df[col] = df.groupby('State')[col].transform(lambda x: x.fillna(x.median()))
    # สุดท้ายถ้ายังว่าง ให้ใช้ค่ากลางรวม
    df[col] = df[col].fillna(df[col].median())

# 3. จัดการ Precipitation (ปริมาณน้ำฝน) แบบเช็คตรรกะความขัดแย้ง
df['Has_Precipitation'] = df['Precipitation(in)'].notna().astype(int)
rain_terms = ['Rain', 'Snow', 'Drizzle', 'Thunderstorm', 'Showers']
is_rainy = df['Weather_Condition'].str.contains('|'.join(rain_terms), case=False, na=False)

# ถ้าชื่อสภาพอากาศบอกว่าฟ้าใส แต่เลขน้ำฝนหาย -> เติม 0
df.loc[~is_rainy & df['Precipitation(in)'].isna(), 'Precipitation(in)'] = 0
# ถ้าชื่อสภาพอากาศบอกว่าฝนตก แต่เลขหาย -> เติมด้วยค่ากลางของพื้นที่/เดือน (ไม่เติม 0)
df['Precipitation(in)'] = df.groupby(['State', 'Start_Month'])['Precipitation(in)'].transform(lambda x: x.fillna(x.median()))
df['Precipitation(in)'] = df['Precipitation(in)'].fillna(0)

# 4. จัดการ Boolean และส่วนที่เหลือ
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].fillna(False)

cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    df[col] = df[col].fillna('Unknown')

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_8820\4226864245.py:31: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include='object').columns


จัดการ Outliers (Clipping)

In [8]:
# จำกัดช่วงค่าที่สมเหตุสมผล
df['Temperature(F)'] = df['Temperature(F)'].clip(-50, 130)
df['Wind_Chill(F)'] = df['Wind_Chill(F)'].clip(-50, 130)
df['Pressure(in)'] = df['Pressure(in)'].clip(26, 32)
df['Visibility(mi)'] = df['Visibility(mi)'].clip(0, 10)

# ตัดค่า Distance ที่กระโดดผิดปกติ (Percentile 99.9)
upper_dist = df['Distance(mi)'].quantile(0.999)
df['Distance(mi)'] = df['Distance(mi)'].clip(upper=upper_dist)

ตรวจสอบ Missing Value รอบสุดท้าย

In [9]:
# 5. ตรวจสอบ Missing Value รอบสุดท้าย (ควรจะเป็น 0 ทั้งหมด)
print("\n--- สรุป Missing Values หลังการ Cleansing ---")
missing_after = df.isnull().sum().sum()
if missing_after == 0:
    print("ยอดเยี่ยม! ไม่พบค่า Missing เหลืออยู่ในชุดข้อมูลแล้ว")
else:
    print(f"ยังคงมีค่า Missing เหลืออยู่ {missing_after} จุด")

# 6. บันทึกข้อมูลที่ Clean แล้วเป็นไฟล์ Parquet (แนะนำมากกว่า CSV เพราะเร็วและไฟล์เล็ก)
output_dir = r"C:\punpun\Big data\US-Traffic-Accident-BigData-Analysis\data\processed"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

output_path = os.path.join(output_dir, "accidents_clean.parquet")
df.to_parquet(output_path, index=False)

print(f"\nบันทึกไฟล์เสร็จเรียบร้อยที่: {output_path}")


--- สรุป Missing Values หลังการ Cleansing ---
ยังคงมีค่า Missing เหลืออยู่ 6911733 จุด

บันทึกไฟล์เสร็จเรียบร้อยที่: C:\punpun\Big data\US-Traffic-Accident-BigData-Analysis\data\processed\accidents_clean.parquet


In [11]:
# ตรวจสอบเฉพาะคอลัมน์ที่เราต้องใช้ทำ Model และ EDA
cols_to_check = [
    'Temperature(F)', 'Wind_Chill(F)', 'Humidity(%)', 'Pressure(in)', 
    'Visibility(mi)', 'Wind_Speed(mph)', 'Precipitation(in)', 
    'Weather_Condition', 'Sunrise_Sunset'
]

print("--- จำนวน Missing คงเหลือในคอลัมน์สำคัญ ---")
print(df[cols_to_check].isnull().sum())

--- จำนวน Missing คงเหลือในคอลัมน์สำคัญ ---
Temperature(F)       0
Wind_Chill(F)        0
Humidity(%)          0
Pressure(in)         0
Visibility(mi)       0
Wind_Speed(mph)      0
Precipitation(in)    0
Weather_Condition    0
Sunrise_Sunset       0
dtype: int64
